# InteractScore: per-residue importance for a protein–ligand interface

Given a processed protein–ligand complex, this notebook computes an **InteractScore** for every residue at the interface by masking the residue one at a time and measuring the cosine similarity between the masked and unmasked complex embeddings produced by the pretrained ATOMICA model.

A **lower cosine similarity** indicates that masking the residue changed the learned complex representation more — i.e. that residue is more important to the interaction. We sort residues by score in **increasing order** (most impactful first) and report the original PDB chain / residue index using `block_to_pdb_indexes`.

Requirements:
- A CUDA-capable GPU (e.g. H100 / A100)
- The `atomica` python environment set up
- Pretrained ATOMICA checkpoint downloaded into `checkpoints/` (see the step below)

## 1. Locate the repository root

We resolve paths relative to the repository root rather than hard-coding absolute paths, so this notebook works for anyone who clones the repo.

In [1]:
import os
from pathlib import Path

def find_repo_root(start: Path) -> Path:
    # Walk upwards looking for a marker that identifies the ATOMICA repo root.
    for parent in [start] + list(start.parents):
        if (parent / "pyproject.toml").exists() and (parent / "src" / "atomica").exists():
            return parent
    raise RuntimeError(f"Could not locate ATOMICA repo root starting from {start}")

REPO_ROOT = find_repo_root(Path(os.getcwd()).resolve())
os.chdir(REPO_ROOT)
print(f"Repo root: {REPO_ROOT}")

Repo root: /n/holylabs/mzitnik_lab/Users/afang/ATOMICA-public


## 2. Download the pretrained ATOMICA checkpoint

Skip this cell if you have already downloaded the pretrained checkpoint (e.g. when running the `1_get_embeddings` tutorial). This requires the `hf` Hugging Face CLI; install it with `pip install -U "huggingface_hub[cli]"` if needed.

In [2]:
ckpt_dir = REPO_ROOT / "checkpoints" / "ATOMICA_checkpoints" / "pretrain"
config_path = ckpt_dir / "pretrain_model_config.json"
weights_path = ckpt_dir / "pretrain_model_weights.pt"

if not (config_path.exists() and weights_path.exists()):
    import subprocess
    subprocess.run(
        [
            "hf", "download", "ada-f/ATOMICA",
            "--repo-type", "model",
            "--local-dir", str(REPO_ROOT / "checkpoints"),
            "--include", "ATOMICA_checkpoints/pretrain/**",
        ],
        check=True,
    )

print(f"config: {config_path}")
print(f"weights: {weights_path}")

config: /n/holylabs/mzitnik_lab/Users/afang/ATOMICA-public/checkpoints/ATOMICA_checkpoints/pretrain/pretrain_model_config.json
weights: /n/holylabs/mzitnik_lab/Users/afang/ATOMICA-public/checkpoints/ATOMICA_checkpoints/pretrain/pretrain_model_weights.pt


## 3. Process the example protein–ligand structure

We use a protein–ligand example from `data/example/example_inputs.csv` — the entry for PDB `6llw`, which is a protein bound to the small-molecule ligand UDP. The processing step converts the `.cif` structure into the block-level interface graph used by ATOMICA and, importantly, also records `block_to_pdb_indexes` so we can map each block back to its original chain / residue number.

In [3]:
import pandas as pd

EXAMPLE_ID = "6llw_A_A_UDP"  # protein–ligand entry

processed_data_path = REPO_ROOT / "data" / "example" / "example_processed_data.parquet"
input_csv = REPO_ROOT / "data" / "example" / "example_inputs.csv"

if not processed_data_path.exists():
    import subprocess
    subprocess.run(
        [
            "python", "-m", "atomica.data.process_pdbs",
            "--data_index_file", str(input_csv),
            "--out_path", str(processed_data_path),
            "--interface_dist_th", "8.0",
            "--fragmentation_method", "PS_300",
        ],
        check=True,
        cwd=REPO_ROOT,
    )

df = pd.read_parquet(processed_data_path)
df[["id"]]

,id
0,6llw_A_A_UDP
1,3i5x_A_B
2,5kl2_A_BC
3,6d1u_A_D
4,2uxq_A_B
5,6hrg_A_A_ZN
6,4yaz_A_A_4BW


## 4. Load the model and the example complex

In [4]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import torch
from atomica.data.dataset import PDBDataset
from atomica.models.prediction_model import PredictionModel

model = PredictionModel.load_from_config_and_weights(str(config_path), str(weights_path))
model = model.to("cuda").eval()

dataset = PDBDataset(str(processed_data_path))
idx = dataset.indexes.index(EXAMPLE_ID)

item = dataset.data[idx]              # full record, including block_to_pdb_indexes
data = dataset[idx]                   # tensor-style graph input used by the model
block_to_pdb = item["block_to_pdb_indexes"]  # block_idx -> "chain_resi"

print(f"Loaded complex {EXAMPLE_ID} with {len(block_to_pdb)} interface blocks.")

Pretrained model params: hidden_size=32,
               edge_size=32, k_neighbors=8, 
               n_layers=4, bottom_global_message_passing=False,
               global_message_passing=True, 
               fragmentation_method=PS_300
Loaded complex 6llw_A_A_UDP with 54 interface blocks.


## 5. Compute the InteractScore for every interface residue

`get_residue_model_scores` loops over every non-global block, masks it, runs the model on both the original and masked versions, and returns the cosine similarity between the two complex-level embeddings. A **lower** cosine similarity means masking that residue had a **larger** effect on the representation — i.e. higher importance.

In [5]:
from atomica.interaction_profiler.interact_score import get_residue_model_scores

cos_distances, block_idx = get_residue_model_scores(model, data)
print(f"Scored {len(cos_distances)} interface residues.")

Scored 54 interface residues.


## 6. Map blocks back to original PDB residues and report

We use `block_to_pdb_indexes` to translate each `block_idx` back to the `chain_residue` from the original PDB file. `segment_ids` lets us distinguish the protein side (segment 0) from the ligand side (segment 1). We print protein residues at the interface sorted by InteractScore in **increasing order** (most impactful first).

In [6]:
segment_ids = data["segment_ids"]

rows = []
for b, score in zip(block_idx, cos_distances):
    pdb_tag = block_to_pdb.get(b)
    if pdb_tag is None:
        continue
    chain, resi = pdb_tag.split("_", 1)
    rows.append({
        "block_idx": b,
        "segment_id": int(segment_ids[b]),
        "chain": chain,
        "residue": resi,
        "interact_score": float(score),
    })

result_df = pd.DataFrame(rows).sort_values("interact_score", ascending=True).reset_index(drop=True)

protein_df = result_df[result_df["segment_id"] == 0]
ligand_df = result_df[result_df["segment_id"] == 1]

print(f"Protein interface residues for {EXAMPLE_ID}, sorted by InteractScore (increasing = more impactful first):\n")
for _, r in protein_df.iterrows():
    print(f"  chain {r['chain']} residue {r['residue']:>4}  (block {int(r['block_idx']):>3})  interact_score = {r['interact_score']:.4f}")

Protein interface residues for 6llw_A_A_UDP, sorted by InteractScore (increasing = more impactful first):

  chain A residue  362  (block  25)  interact_score = 0.9854
  chain A residue  340  (block  15)  interact_score = 0.9884
  chain A residue  344  (block  19)  interact_score = 0.9945
  chain A residue  360  (block  23)  interact_score = 0.9951
  chain A residue  339  (block  14)  interact_score = 0.9960
  chain A residue  345  (block  20)  interact_score = 0.9963
  chain A residue  361  (block  24)  interact_score = 0.9963
  chain A residue  278  (block   8)  interact_score = 0.9964
  chain A residue  358  (block  21)  interact_score = 0.9967
  chain A residue  364  (block  27)  interact_score = 0.9970
  chain A residue  341  (block  16)  interact_score = 0.9973
  chain A residue  305  (block  12)  interact_score = 0.9977
  chain A residue  363  (block  26)  interact_score = 0.9977
  chain A residue  277  (block   7)  interact_score = 0.9978
  chain A residue  306  (block  13)  in

## 7. (Optional) Also print the ligand-side blocks

In [7]:
print(f"Ligand-side blocks for {EXAMPLE_ID}, sorted by InteractScore (increasing):\n")
for _, r in ligand_df.iterrows():
    print(f"  chain {r['chain']} residue {r['residue']:>4}  (block {int(r['block_idx']):>3})  interact_score = {r['interact_score']:.4f}")

Ligand-side blocks for 6llw_A_A_UDP, sorted by InteractScore (increasing):

  chain A residue  900  (block  48)  interact_score = 0.9978
  chain A residue  900  (block  47)  interact_score = 0.9979
  chain A residue  900  (block  36)  interact_score = 0.9979
  chain A residue  900  (block  49)  interact_score = 0.9981
  chain A residue  900  (block  52)  interact_score = 0.9981
  chain A residue  900  (block  39)  interact_score = 0.9982
  chain A residue  900  (block  50)  interact_score = 0.9984
  chain A residue  900  (block  54)  interact_score = 0.9985
  chain A residue  900  (block  38)  interact_score = 0.9985
  chain A residue  900  (block  35)  interact_score = 0.9986
  chain A residue  900  (block  46)  interact_score = 0.9989
  chain A residue  900  (block  33)  interact_score = 0.9990
  chain A residue  900  (block  43)  interact_score = 0.9990
  chain A residue  900  (block  37)  interact_score = 0.9991
  chain A residue  900  (block  44)  interact_score = 0.9992
  chain A